# Quantization

এই notebook দুটি demo চালায়, দুটিই সাধারণ PyTorch tensor ops-এ স্ক্র্যাচ থেকে বাস্তবায়িত (কোনো বাইরের quantization library নেই):

1. একটি র্যান্ডম weight matrix-এর **symmetric INT8 এবং INT4** quantization/dequantization — reconstruction error (MSE, max absolute error) এবং প্রকৃত memory footprint হ্রাস মাপা — সাথে একটি FP8 (E4M3-style) floating-point quantizer একইভাবে সিমুলেটেড, যাতে INT8-এর স্থির step size-কে একই 8-bit বাজেটে FP8-এর floating exponent-এর সাথে সরাসরি তুলনা করা যায়।
2. একটি সরলীকৃত **AWQ-style demo**: naive uniform INT4 quantization বনাম একটি activation-aware স্কিম যা top-k% উচ্চতম-magnitude-activation columns-কে পূর্ণ নির্ভুলতায় রাখে, মিলানো কার্যকর গড় bit-width-এ তুলনা করা হয়।

**চালানোর নিয়ম:** উপরের দিক থেকে নিচের দিকে প্রতিটি cell চালান; প্রতিটি সেকশনের demo তার নিজ cell-এর শেষেই চলে।

In [ ]:
import torch

torch.manual_seed(0)

## 1. স্ক্র্যাচ থেকে symmetric quantization / dequantization

প্রথম demo একটি র্যান্ডম weight matrix-এ **INT8 এবং INT4** symmetric quantization/dequantization বাস্তবায়ন করে (README section 2-এর ফর্মুলা), reconstruction error (MSE ও max absolute error) এবং প্রকৃত memory footprint হ্রাস মাপে। তারপর একটি **FP8 (E4M3-style)** floating-point quantizer একইভাবে সিমুলেট করা হয় (README section 6), যাতে একই 8-bit বাজেটে দুটি ভিন্ন বিতরণে INT8 ও FP8-এর মধ্যে তুলনা দেখা যায় — সংকীর্ণ Gaussian-সদৃশ weights বনাম বিরল বড়-magnitude outliers-সহ wide-dynamic-range tensor।

In [ ]:
# ---------------------------------------------------------------------------
# 1. স্ক্র্যাচ থেকে symmetric quantization / dequantization
# ---------------------------------------------------------------------------

def quantize_symmetric(x, bits):
    """Uniform symmetric quantization (README section 2).

    scale = max(|x|) / (2^(bits-1) - 1)
    q     = round(x / scale), representable integer range-এর মধ্যে clamp করা
    """
    qmax = 2 ** (bits - 1) - 1     # যেমন INT8-এর জন্য 127, INT4-এর জন্য 7
    scale = x.abs().max() / qmax
    q = torch.clamp(torch.round(x / scale), -qmax, qmax)
    return q, scale


def dequantize_symmetric(q, scale):
    return q * scale


def quantize_float_format(x, exp_bits, mantissa_bits):
    """Low-bit IEEE-754-style floating-point format (যেমন FP8 E4M3 হল exp_bits=4,
    mantissa_bits=3) সিমুলেট করে -- README section 6। quantize_symmetric-এর স্থির
    step size-এর মতো নয়, প্রতিটি মান নিজের exponent ধরে রাখে (exp_bits যা
    উপস্থাপন করতে পারে সেখানে clamp করা) এবং শুধু এর mantissa-ই mantissa_bits
    নির্ভুলতায় বৃত্তাকার হয়। এই সিমুলেটর subnormals/inf/NaN-এর সামলানো এড়িয়ে যায়
    (এখানে reconstruction-error তুলনায় অপ্রাসঙ্গিক) কিন্তু মূল trade-off পুনরুৎপাদন
    করে: একটি floating format তার সীমিত bits-গুলো RELATIVE নির্ভুলতায় ব্যয় করে
    (যেকোনো magnitude-এ একই সংখ্যক significant bits), যেখানে INT8/INT4-এর স্থির
    step size সেগুলো একটি range জুড়ে ABSOLUTE নির্ভুলতায় ব্যয় করে।
    """
    sign = torch.sign(x)
    x_abs = torch.clamp(x.abs(), min=1e-12)   # log2(0) এড়ানো
    bias = 2 ** (exp_bits - 1) - 1
    exponent = torch.clamp(torch.floor(torch.log2(x_abs)), -bias + 1, bias)
    mantissa_levels = 2 ** mantissa_bits
    frac = torch.clamp(x_abs / (2.0 ** exponent) - 1.0, 0.0, 1.0)   # [0, 1)-এ
    frac_q = torch.round(frac * mantissa_levels) / mantissa_levels
    x_hat = sign * (2.0 ** exponent) * (1.0 + frac_q)
    return torch.where(x.abs() == 0, torch.zeros_like(x_hat), x_hat)


def quantization_error_demo():
    print("=" * 70)
    print("1. SYMMETRIC INT8 / INT4 QUANTIZATION FROM SCRATCH")
    print("=" * 70)

    # একটি Transformer FFN projection (Phase 02 Lesson 5-এর FFN sublayer) এর জন্য
    # বাস্তব-সদৃশ আকৃতির একটি weight matrix, একটি বাস্তব প্রশিক্ষিত layer-এর
    # বিতরণের মোটামুটি মিল (ছোট, বেশিরভাগ Gaussian -- পয়েন্টটি তুলে ধরতে এর চেয়ে
    # জটিল কিছুর দরকার নেই)।
    rows, cols = 512, 512
    W = torch.randn(rows, cols) * 0.02
    num_elements = rows * cols
    fp32_bytes = num_elements * 4

    print(f"weight matrix shape: {tuple(W.shape)}  ({num_elements:,} elements)")
    print(f"original dtype: float32  ({fp32_bytes:,} bytes = {fp32_bytes / 1024:.1f} KB)\n")

    print(f"{'scheme':>10}{'bits':>7}{'MSE':>16}{'max abs err':>16}{'bytes':>12}{'reduction':>12}")
    results = {}
    for bits, packed_bytes_per_elem in [(8, 1.0), (4, 0.5)]:
        q, scale = quantize_symmetric(W, bits)
        W_hat = dequantize_symmetric(q, scale)

        mse = torch.mean((W - W_hat) ** 2).item()
        max_abs_err = (W - W_hat).abs().max().item()
        # বাস্তব বাস্তবায়নে INT4 মানগুলো প্রতি-বাইটে দুইটি করে প্যাক করা হয়;
        # per-tensor scale (একটি একক float32) হল নগণ্য overhead।
        quantized_bytes = num_elements * packed_bytes_per_elem
        reduction = fp32_bytes / quantized_bytes

        results[bits] = (mse, max_abs_err, quantized_bytes, reduction)
        print(f"{'INT' + str(bits):>10}{bits:>7}{mse:>16.3e}{max_abs_err:>16.5f}"
              f"{quantized_bytes:>12,.0f}{reduction:>11.1f}x")

    mse8, _, bytes8, red8 = results[8]
    mse4, _, bytes4, red4 = results[4]
    print(f"\n-> INT8 shrinks this layer's weights by {red8:.0f}x (from "
          f"{fp32_bytes/1024:.1f} KB to {bytes8/1024:.1f} KB) with a tiny MSE of "
          f"{mse8:.2e}.")
    print(f"-> INT4 shrinks it by {red4:.0f}x, but the MSE rises to {mse4:.2e} -- "
          f"{mse4/mse8:.1f}x worse than INT8 -- because only "
          f"{2**3 - 1} distinct positive integer levels are available to represent")
    print("   the entire range of weight values, versus 127 for INT8. This is exactly")
    print("   the accuracy cliff GPTQ and AWQ exist to soften (README sections 3-4).")

    # --- FP8 (E4M3-style) একই 8-bit বাজেটে INT8-এর পাশাপাশি (README section 6),
    #     দুটি ভিন্ন বিতরণে -- দেখাতে যে FP8 শুধু "INT8-এর ভিন্ন নাম" নয়; কোন
    #     ফরম্যাট জেতে তা bit সংখ্যা নয়, ডেটার আকৃতির উপর নির্ভর করে:
    #       (a) উপরের মতো একই সংকীর্ণ, মোটামুটি Gaussian weight matrix
    #       (b) কয়েকটি বড় outlier-সহ wide-dynamic-range tensor, যেমন বাস্তব
    #           Transformer ACTIVATIONS-এর আকৃতি থাকে
    #           (নিচের AWQ demo-ও যে একই ঘটনা কাজে লাগায়) ---
    def fp8_vs_int8(tensor, label):
        q8, scale8 = quantize_symmetric(tensor, bits=8)
        mse_int8 = torch.mean((tensor - dequantize_symmetric(q8, scale8)) ** 2).item()
        W_hat_fp8 = quantize_float_format(tensor, exp_bits=4, mantissa_bits=3)
        mse_fp8 = torch.mean((tensor - W_hat_fp8) ** 2).item()
        winner = "FP8" if mse_fp8 < mse_int8 else "INT8"
        print(f"{label:>34}{mse_int8:>16.3e}{mse_fp8:>16.3e}{winner:>16}")
        return mse_int8, mse_fp8

    outlier_tensor = torch.randn(rows, cols) * 0.02
    num_outliers = max(1, int(outlier_tensor.numel() * 0.001))
    flat = outlier_tensor.flatten()
    flat[torch.randperm(flat.numel())[:num_outliers]] *= 200.0   # একটি বিরল, বড়-magnitude tail

    print(f"\n{'distribution':>34}{'INT8 MSE':>16}{'FP8(E4M3) MSE':>16}{'lower error':>16}")
    fp8_vs_int8(W, "narrow, ~Gaussian (this layer's weights)")
    fp8_vs_int8(outlier_tensor, "wide-range, rare large outliers")

    print("\n-> Same 8-bit budget, same tensor SHAPE, opposite winner depending on the data's")
    print("   dynamic range. INT8's fixed step size is calibrated to the single largest")
    print("   value in the tensor (max(|x|)/127, README section 2) -- on a narrow, ")
    print("   well-behaved distribution that step is already fine-grained, so INT8 wins.")
    print("   Introduce a few rare, large-magnitude outliers and that SAME calibration")
    print("   rule blows the step size up for the whole tensor, crushing resolution")
    print("   everywhere else -- while FP8's floating exponent keeps giving every value")
    print("   the same RELATIVE precision regardless of magnitude, so it barely notices")
    print("   the outliers. This is exactly why FP8 is the more common choice for")
    print("   ACTIVATIONS (which have real outlier channels -- the AWQ demo below is")
    print("   built around this same phenomenon) while weights, often narrower-range,")
    print("   can do just as well or better in INT8. Either format needs the same 1")
    print("   byte/element to store, and the same Tensor Core support (Lesson 1 section 7)")
    print("   to realize a matching compute speedup, not just the memory saving.")


quantization_error_demo()

## 2. সরলীকৃত AWQ-style demo: activation-salient columns রক্ষা করা

দ্বিতীয় demo-টি naive uniform INT4 quantization-কে একটি AWQ-style mixed-precision স্কিমের সাথে তুলনা করে (README section 4): কৃত্রিম per-column "activation magnitude" statistics-এর উপর ভিত্তি করে top-k% salient columns-গুলো fp32-এ রাখা হয় এবং বাকিগুলো INT4-তে quantized হয়। তুলনাটি মেলানো **কার্যকর** গড় bit-width-এ হয়, এবং প্রতিটি column-এর reconstruction error-কে সেই column-কে গুণ করা activation-এর magnitude দিয়ে ওজন করা হয় — যাতে error-টা layer output-এর (y = x @ W) উপর প্রভাব প্রতিফলিত করে, শুধু কাঁচা weight error নয়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. একটি সরলীকৃত AWQ-style demo: activation-salient columns রক্ষা করা
# ---------------------------------------------------------------------------

def quantize_matrix_int4(W):
    """একটি সম্পূর্ণ matrix-কে uniform INT4-তে quantize করে (naive RTN, per-tensor scale)।"""
    q, scale = quantize_symmetric(W, bits=4)
    return dequantize_symmetric(q, scale)


def awq_style_demo():
    print("\n" + "=" * 70)
    print("2. AWQ-STYLE DEMO: PROTECTING ACTIVATION-SALIENT COLUMNS")
    print("=" * 70)

    rows, cols = 256, 256
    W = torch.randn(rows, cols) * 0.02

    # Calibration-data activation statistics সিমুলেট করা: বেশিরভাগ input channel
    # (W-এর columns, যেহেতু y = x @ W column j-কে input channel j দিয়ে গুণ করে)
    # সাধারণ magnitude-র, কিন্তু একটি ছোট ভগ্নাংশ ধারাবাহিকভাবে বড় -- ঠিক সেই
    # "outlier channel" ঘটনা যা AWQ এবং LLM.int8() দুটোই কাজে লাগায়।
    activation_magnitude = torch.abs(torch.randn(cols))
    salient_fraction = 0.05
    num_salient = max(1, int(cols * salient_fraction))
    # কয়েকটি সত্যিই বড়-magnitude activation channel injected করা যাতে
    # salient-column প্রভাব বাস্তব এবং মাপার মতো হয়, শুধু noise না।
    outlier_cols = torch.randperm(cols)[:num_salient]
    activation_magnitude[outlier_cols] *= 15.0

    print(f"weight matrix shape: {tuple(W.shape)}")
    print(f"salient columns: top {salient_fraction*100:.0f}% by simulated activation "
          f"magnitude = {num_salient} of {cols} columns\n")

    # --- (a) পুরো matrix-এর naive uniform INT4 quantization ---
    W_hat_naive = quantize_matrix_int4(W)

    # Output-error তুলনাটি অর্থবহ করতে (শুধু কাঁচা weight error নয়), প্রতিটি
    # column-এর reconstruction error-কে ওজন করা হয় যে column-টিকে গুণ করা
    # activationsগুলো আসলে কত বড় -- এটি approximation করে quantization error-র
    # LAYER-এর OUTPUT-এ (y = x @ W) প্রভাব, যেহেতু বড় activations-সহ একটি column-এর
    # একটি স্থির weight error y-তে আনুপাতিকভাবে বড় error তৈরি করে।
    def activation_weighted_error(W_orig, W_hat, act_mag):
        per_col_sq_err = ((W_orig - W_hat) ** 2).mean(dim=0)     # (cols,)
        weighted = (per_col_sq_err * act_mag ** 2).sum() / (act_mag ** 2).sum()
        return weighted.item()

    naive_output_err = activation_weighted_error(W, W_hat_naive, activation_magnitude)
    naive_raw_mse = torch.mean((W - W_hat_naive) ** 2).item()

    # --- (b) AWQ-style: top-k% salient columns fp32-এ রাখা, বাকিগুলো quantize করা ---
    salient_mask = torch.zeros(cols, dtype=torch.bool)
    salient_mask[outlier_cols] = True

    W_hat_awq = W.clone()
    non_salient = W[:, ~salient_mask]
    q, scale = quantize_symmetric(non_salient, bits=4)
    W_hat_awq[:, ~salient_mask] = dequantize_symmetric(q, scale)
    # salient columns (W_hat_awq[:, salient_mask]) fp32-এ অপরিবর্তিত রাখা হয়

    awq_output_err = activation_weighted_error(W, W_hat_awq, activation_magnitude)
    awq_raw_mse = torch.mean((W - W_hat_awq) ** 2).item()

    # কার্যকর গড় bit-width: num_salient columns 32 bits-এ থাকে,
    # অবশিষ্ট (cols - num_salient) columns 4 bits-এ নামে।
    naive_effective_bits = 4.0
    awq_effective_bits = (num_salient * 32 + (cols - num_salient) * 4) / cols

    print(f"{'scheme':>28}{'effective bits/weight':>24}{'raw weight MSE':>18}"
          f"{'activation-weighted err':>26}")
    print(f"{'naive uniform INT4':>28}{naive_effective_bits:>24.2f}"
          f"{naive_raw_mse:>18.3e}{naive_output_err:>26.3e}")
    print(f"{'AWQ-style (protect top ' + f'{salient_fraction*100:.0f}%)':>28}"
          f"{awq_effective_bits:>24.2f}{awq_raw_mse:>18.3e}{awq_output_err:>26.3e}")

    print(f"\n-> Protecting just the top {salient_fraction*100:.0f}% of columns raises the "
          f"effective bit-width only from {naive_effective_bits:.2f} to "
          f"{awq_effective_bits:.2f} bits/weight (still overwhelmingly INT4), yet the "
          f"activation-weighted output error drops by "
          f"{naive_output_err / awq_output_err:.1f}x (from {naive_output_err:.2e} to "
          f"{awq_output_err:.2e}).")
    print("-> This is the AWQ insight made concrete: a small minority of weight columns,")
    print("   the ones multiplied by large-magnitude activations, dominate the layer's")
    print("   OUTPUT error under quantization -- so protecting just those few columns")
    print("   buys most of full-precision's accuracy at almost none of its memory cost.")
    print(f"   (For reference, raw unweighted weight MSE only improves by "
          f"{naive_raw_mse / awq_raw_mse:.2f}x -- the benefit is concentrated exactly where")
    print("   the activations are large, which is the whole point.)")


awq_style_demo()

In [ ]:
def main():
    quantization_error_demo()
    awq_style_demo()


main()